In [1]:
print('hello')
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.pyplot import rc_context
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import scanpy as sc
from matplotlib import rc_context

h5ad_file = "/home/catherine/phd/projects/termites/data/znev/combined_no_norm_clustered.h5ad"
adata = sc.read_h5ad(h5ad_file)


hello


In [4]:
adata.var_names.unique

<bound method Index.unique of Index(['Znev00005426', 'Znev00005427', 'Znev00005428', 'Znev00005429',
       'Znev00005430', 'Znev00005431', 'Znev00005432', 'Znev00005433',
       'Znev00005434', 'Znev00005435',
       ...
       'Znev00014242', 'Znev00014267', 'Znev00014257', 'Znev00014237',
       'Znev00014238', 'Znev00014255', 'Znev00014250', 'Znev00014246',
       'Znev00014270', 'Znev00014265'],
      dtype='object', length=14272)>

In [9]:
# take the results from the mapping of the znev transcripts to the dmel proteins from blastx search and map the transcripts from the dmel protein
# to the dmel gene, 

from collections import OrderedDict
import re

def parse_blastx(file_path):
    """Parse the BLASTX output file, keeping the first unique occurrence of trimmed query IDs."""
    id_mapping = OrderedDict()
    
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            trimmed_id = parts[0][:-3]  # Remove last three characters
            match_id = parts[1]  # The second column (e.g., FBpp0422979)
            
            if trimmed_id not in id_mapping:
                id_mapping[trimmed_id] = match_id
    
    return id_mapping

def parse_fasta(file_path):
    """Parse the FASTA file to map FBpp IDs to their parent IDs."""
    fbpp_to_parent = {}
    
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('>'):
                match = re.search(r'>(FBpp\d+).*?parent=([^;]+)', line)
                if match:
                    fbpp_id = match.group(1)
                    parent_id = match.group(2).split(',')[0]  # Get the first parent ID
                    fbpp_to_parent[fbpp_id] = parent_id
    
    return fbpp_to_parent

def map_ids(blastx_file, fasta_file):
    """Map trimmed BLASTX query IDs to their corresponding parent IDs from the FASTA file."""
    id_mapping = parse_blastx(blastx_file)
    fbpp_to_parent = parse_fasta(fasta_file)
    
    result_mapping = {}
    for trimmed_id, fbpp_id in id_mapping.items():
        parent_id = fbpp_to_parent.get(fbpp_id, 'Unknown')
        result_mapping[trimmed_id] = parent_id
    
    return result_mapping

if __name__ == "__main__":
    blastx_file = "/home/catherine/phd/projects/termites/code/znev_analysis/fly_find_homologs/all_dmel_gene_to_protein_blast_znev.txt"  
    fasta_file = "/home/catherine/phd/projects/termites/code/znev_analysis/fly_find_homologs/dmel-all-translation-r6.62.fasta" 
    
    result = map_ids(blastx_file, fasta_file)
    print(result)


{'Znev00005427': 'FBgn0030731', 'Znev00005429': 'FBgn0025186', 'Znev00005431': 'FBgn0031497', 'Znev00005432': 'FBgn0040324', 'Znev00005444': 'FBgn0039013', 'Znev00005448': 'FBgn0037770', 'Znev00005451': 'FBgn0001104', 'Znev00005452': 'FBgn0053181', 'Znev00005453': 'FBgn0030674', 'Znev00005455': 'FBgn0039358', 'Znev00005458': 'FBgn0016920', 'Znev00005459': 'FBgn0002440', 'Znev00005459-': 'FBgn0002440', 'Znev00005460': 'FBgn0053181', 'Znev00005465': 'FBgn0020309', 'Znev00005467': 'FBgn0032295', 'Znev00005468': 'FBgn0031375', 'Znev00005470': 'FBgn0000520', 'Znev00005471': 'FBgn0025874', 'Znev00005472': 'FBgn0000286', 'Znev00005473': 'FBgn0020309', 'Znev00005474': 'FBgn0032979', 'Znev00005475': 'FBgn0046685', 'Znev00005476': 'FBgn0010611', 'Znev00005477': 'FBgn0039914', 'Znev00005479': 'FBgn0014007', 'Znev00005482': 'FBgn0031985', 'Znev00005483': 'FBgn0265192', 'Znev00005484': 'FBgn0044826', 'Znev00005485': 'FBgn0284255', 'Znev00005486': 'FBgn0003011', 'Znev00005487': 'FBgn0005658', 'Znev0

In [10]:
# now I want to annotate the gene_ids with the fly gene 
adata



AnnData object with n_obs × n_vars = 24252 × 14272
    obs: 'sample#', 'caste', 'n_genes_by_counts', 'total_counts', 'leiden', 'samap_annot', 'labels_only_znev_fabio', 'leiden_fabio', 'leiden_dmel_fabio', 'znev_leiden_dmel_fabio'
    var: 'gene_ids', 'feature_types', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'annotate_cells_colors', 'annotations_1_colors', 'caste_colors', 'hvg', 'leiden', 'leiden_colors', 'leiden_dmel_fabio_colors', 'leiden_fabio_colors', 'log1p', 'neighbors', 'new_clusters_colors', 'pca', 'rank_genes_groups', 'samap_annot_colors', 'umap', 'znev_leiden_dmel_fabio_colors'
    obsm: 'X_pca', 'X_umap', 'X_umap_fabio'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [11]:
adata.var['dmel_gene_ortho'] = adata.var_names.map(result).fillna('NA')

In [15]:
adata.var['dmel_gene_ortho']
print((adata.var['dmel_gene_ortho'] != 'NA').sum())

8754


In [16]:
# ok now I want to save this 
adata.write(h5ad_file)